# Employee Clustering (Supervised)Converted from `src/clustering.py`---

**Beschreibung:** Employee Clustering — Leakage-Free Unsupervised Analysis=========================================================DATA SCIENCE NOTE: This clustering is leakage-free because:1. Unsupervised Learning — kein Target-Variable, kein Zirkelschluss2. Features sind direkte Aggregationen aus Rohdaten (kein O-Score)3. No rank functions that could contaminate Train/Testn4. No Train/Test split needed (clustering = descriptive, not predictivetiv)Note: An earlier ML classifier (ml_model_o.py) was entfernt, da er zweiData-Science-Regeln verletzt hat:- Tautological model: Target = deterministic function der Features- Data Leakage: Rank-based features across the full dataset before Train/Test-SplitDer O-Score wird nun direkt als regelbasierter Composite-Score (o_score.py) bereitgestellt.

In [9]:
import os
import sys
import warnings

# Suppress unnecessary warnings (useful during experimentation)
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

# Clustering & evaluation
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

# ─── Project structure ───
# If you're running this in a Jupyter notebook, __file__ usually doesn't exist.
# Use one of these safer approaches instead:

# Option A: Hard-coded relative paths (most reliable in notebooks)
PROJECT_ROOT = Path.cwd().resolve()  # current working directory

# Option B: If you really want parent of parent (uncomment if needed)
# try:
#     PROJECT_ROOT = Path(__file__).resolve().parent.parent
# except NameError:
#     PROJECT_ROOT = Path.cwd().resolve()

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "issues_snapshot.csv"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"

# Create directories if they don't exist
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Quick debug output – very helpful when things go wrong
print("Project setup:")
print(f"  Current working dir: {Path.cwd().absolute()}")
print(f"  Project root:        {PROJECT_ROOT.absolute()}")
print(f"  Raw data file:       {DATA_RAW}")
print(f"  Exists?              {DATA_RAW.exists()}")
print(f"  Processed dir:       {DATA_PROCESSED}")
print(f"  Models dir:          {MODELS_DIR}")

Project setup:
  Current working dir: /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks
  Project root:        /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks
  Raw data file:       /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw/issues_snapshot.csv
  Exists?              False
  Processed dir:       /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/processed
  Models dir:          /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/models


## SCHRITT 1: Feature Engineering (leakage-frei)

In [10]:
import pandas as pd

def build_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregiere pro Mitarbeiter — nur direkte Aggregationen,
    kein Overall-Score, keine Rang-Funktionen.
    
    Returns:
        DataFrame mit einem Mitarbeiter pro Zeile
    """
    print("Building features...")
    
    # Nur Zeilen mit Assignee
    df = df[df['issue_assignee'].notna()].copy()
    
    # ─── Basis-Aggregationen ───
    agg = df.groupby('issue_assignee').agg(
        ticket_count=('id', 'count'),
        median_time_sec=('wf_total_time', 'median'),
        std_time_sec=('wf_total_time', 'std'),
        avg_steps=('processing_steps', 'mean'),
        avg_comments=('issue_comments_count', 'mean'),
        reopen_rate=('wfe_reopened', lambda x: (x > 0).mean()),
        success_rate=('issue_resolution', lambda x: (x == 'Done').mean()),
        first_touch_rate=('turn', lambda x: (x == 1).mean()),
    ).reset_index()
    
    # Zeit in Stunden umrechnen
    agg['median_time_hours'] = agg['median_time_sec'] / 3600.0
    agg['std_time_hours']   = agg['std_time_sec']   / 3600.0
    agg.drop(columns=['median_time_sec', 'std_time_sec'], inplace=True)
    
    # ─── Priority-Mix ───
    priority_agg = df.groupby('issue_assignee').apply(
        lambda g: pd.Series({
            'pct_high': (g['issue_priority'].isin(['High', 'Highest', 'Critical', 'Blocker'])).mean(),
            'pct_low':  (g['issue_priority'].isin(['Low', 'Lowest', 'Minor'])).mean(),
        })
    ).reset_index()
    
    # ─── Type-Mix ───
    type_agg = df.groupby('issue_assignee').apply(
        lambda g: pd.Series({
            'pct_hd_service': (g['issue_type'] == 'HD Service').mean(),
        })
    ).reset_index()
    
    # ─── Zusammenführen ───
    result = agg.merge(priority_agg, on='issue_assignee', how='left')
    result = result.merge(type_agg,    on='issue_assignee', how='left')
    
    # Filter: nur Mitarbeiter mit mindestens 10 Tickets
    result = result[result['ticket_count'] >= 10].reset_index(drop=True)
    
    print(f"  Mitarbeiter nach Filter (≥ 10 Tickets): {len(result)}")
    
    return result

## SCHRITT 2: Preprocessing

In [11]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

def preprocess(feature_df: pd.DataFrame, feature_cols: list) -> tuple:
    """
    Scale features, fill NaN with median, remove zero-variance columns.
    Log-transformiert stark schiefe Features (Zeit, Ticket-Anzahl).
    
    Args:
        feature_df:   DataFrame mit Mitarbeiter-Features (eine Zeile pro Person)
        feature_cols: Liste der zu verwendenden Feature-Spalten
    
    Returns:
        tuple: (X_scaled: np.ndarray, scaler: RobustScaler, active_cols: list)
    """
    print("Preprocessing features...")
    
    # Nur die gewünschten Spalten + Kopie
    X = feature_df[feature_cols].copy()
    
    # 1. NaN → Median (robust gegen Ausreißer)
    for col in X.columns:
        if X[col].isna().any():
            median_val = X[col].median()
            X[col].fillna(median_val, inplace=True)
            print(f"   → {col}: {X[col].isna().sum()} NaN → mit Median {median_val:.3f} gefüllt")
    
    # 2. Log-Transformation für stark rechtsschiefe Features
    skewed_cols = ['ticket_count', 'median_time_hours', 'std_time_hours',
                   'avg_steps', 'avg_comments']
    log_transformed = []
    
    for col in skewed_cols:
        if col in X.columns:
            # log1p = log(1 + x) → sicher bei 0-Werten
            X[col] = np.log1p(X[col])
            log_transformed.append(col)
    
    # 3. Zero-Variance-Spalten entfernen (bringen nichts für Clustering/ML)
    zero_var_cols = X.columns[X.std() == 0].tolist()
    if zero_var_cols:
        print(f" Entferne Zero-Variance columns: {zero_var_cols}")
        X.drop(columns=zero_var_cols, inplace=True)
    
    active_cols = X.columns.tolist()
    
    # 4. Robust Scaling (median-basiert, gut gegen Ausreißer)
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    
    # ─── Zusammenfassung ───
    print(f" Features nach Preprocessing: {len(active_cols)} Spalten, "
          f"{len(X_scaled)} Mitarbeiter")
    if log_transformed:
        print(f" Log-transformiert: {log_transformed}")
    if zero_var_cols:
        print(f" Verbleibende Features: {active_cols}")
    
    return X_scaled, scaler, active_cols

## SCHRITT 3: Dimensionsreduktion

In [12]:
import numpy as np
from sklearn.decomposition import PCA

def reduce_dimensions(X_scaled: np.ndarray) -> dict:
    """
    PCA + UMAP (falls verfügbar) für 2D-Visualisierung und Clustering-Vorbereitung.
    
    Args:
        X_scaled: skaliertes numpy-Array (n_samples, n_features)
    
    Returns:
        dict mit:
          - 'pca':        PCA-transformierte 2D-Daten
          - 'pca_model':  das PCA-Objekt
          - 'umap':       UMAP- oder PCA-Fallback 2D-Daten
          - 'umap_model': UMAP- oder PCA-Objekt
          - 'use_umap':   bool, ob UMAP erfolgreich war
    """
    result = {}
    
    # ─── PCA (immer verfügbar) ───
    pca = PCA(n_components=2, random_state=42)
    X_pca = pca.fit_transform(X_scaled)
    
    result['pca'] = X_pca
    result['pca_model'] = pca
    
    explained = pca.explained_variance_ratio_.sum() * 100
    print(f"  PCA: {explained:.1f}% Varianz erklärt durch 2 Komponenten")
    
    # ─── UMAP (besser bei nicht-linearen Strukturen) ───
    try:
        import umap
        reducer = umap.UMAP(
            n_components=2,
            random_state=42,
            n_neighbors=15,
            min_dist=0.1
        )
        X_umap = reducer.fit_transform(X_scaled)
        
        result['umap'] = X_umap
        result['umap_model'] = reducer
        result['use_umap'] = True
        
        print("  UMAP: erfolgreich berechnet")
        
    except (ImportError, Exception) as ex:
        print(f"  UMAP nicht verfügbar ({ex}), verwende PCA als Fallback")
        result['umap'] = X_pca
        result['umap_model'] = pca
        result['use_umap'] = False
    
    return result

## SCHRITT 4: Clustering-Algorithmen vergleichen

In [13]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

def run_clustering_comparison(X_scaled: np.ndarray) -> tuple:
    """
    Testet KMeans, GaussianMixture, AgglomerativeClustering für k=2..6
    sowie HDBSCAN (falls verfügbar).
    Berechnet Silhouette, Davies-Bouldin und Calinski-Harabasz Scores.
    
    Returns:
        tuple: (df_results: pd.DataFrame mit Metriken, results: list mit vollen Objekten)
    """
    results = []
    k_values = [2, 3, 4, 5, 6]
    
    print("\n  Algorithmen-Vergleich:")
    print(f"  {'Algo':<20} {'k':>3} {'Silhouette':>12} {'DB-Score':>10} {'CH-Score':>10}")
    print("  " + "-" * 58)
    
    for k in k_values:
        algorithms = {
            'KMeans': KMeans(n_clusters=k, random_state=42, n_init=10),
            'GaussianMixture': GaussianMixture(n_components=k, random_state=42),
            'Agglomerative': AgglomerativeClustering(n_clusters=k, linkage='ward'),
        }
        
        for algo_name, model in algorithms.items():
            try:
                labels = model.fit_predict(X_scaled)
                n_clusters_found = len(set(labels))
                
                if n_clusters_found < 2:
                    print(f"  {algo_name:<20} {k:>3} → nur {n_clusters_found} Cluster → übersprungen")
                    continue
                
                sil = silhouette_score(X_scaled, labels)
                db = davies_bouldin_score(X_scaled, labels)
                ch = calinski_harabasz_score(X_scaled, labels)
                
                results.append({
                    'algorithm': algo_name,
                    'k': k,
                    'silhouette': round(sil, 4),
                    'davies_bouldin': round(db, 4),
                    'calinski_harabasz': round(ch, 2),
                    'labels': labels,
                    'model': model,
                })
                
                print(f"  {algo_name:<20} {k:>3} {sil:>12.4f} {db:>10.4f} {ch:>10.2f}")
                
            except Exception as ex:
                print(f"  {algo_name:<20} {k:>3} FEHLER: {ex}")
    
    # ─── HDBSCAN (density-based, keine feste k-Vorgabe) ───
    try:
        import hdbscan
        hdb = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=3)
        labels_hdb = hdb.fit_predict(X_scaled)
        
        n_noise = (labels_hdb == -1).sum()
        unique_labels = set(labels_hdb)
        n_clusters_hdb = len(unique_labels) - (1 if -1 in unique_labels else 0)
        
        mask = labels_hdb != -1
        if mask.sum() > n_clusters_hdb and n_clusters_hdb >= 2:
            sil_hdb = silhouette_score(X_scaled[mask], labels_hdb[mask])
            db_hdb = davies_bouldin_score(X_scaled[mask], labels_hdb[mask])
            ch_hdb = calinski_harabasz_score(X_scaled[mask], labels_hdb[mask])
            
            results.append({
                'algorithm': 'HDBSCAN',
                'k': n_clusters_hdb,
                'silhouette': round(sil_hdb, 4),
                'davies_bouldin': round(db_hdb, 4),
                'calinski_harabasz': round(ch_hdb, 2),
                'labels': labels_hdb,
                'model': hdb,
                'noise_count': n_noise,
            })
            
            print(f"  {'HDBSCAN':<20} {n_clusters_hdb:>3} {sil_hdb:>12.4f} {db_hdb:>10.4f} {ch_hdb:>10.2f}  (Rauschen: {n_noise})")
        else:
            print(f"  HDBSCAN: zu wenige valide Cluster ({n_clusters_hdb}), nicht auswertbar")
            
    except (ImportError, Exception) as ex:
        print(f"  HDBSCAN nicht verfügbar: {ex}")
    
    # ─── Ergebnisse als DataFrame (ohne Model-Objekte) ───
    df_results = pd.DataFrame([
        {k: v for k, v in r.items() if k not in ('labels', 'model')}
        for r in results
    ])
    
    return df_results, results

## STEP 5: Select Best Configuration

In [15]:
import pandas as pd

def select_best(comparison_results: list) -> dict:
    """
    Wähle die beste Clustering-Konfiguration anhand des Silhouette-Scores.
    Filtert degenerierte Lösungen aus (Cluster mit < ~2–3% der Mitarbeiter).
    
    Args:
        comparison_results: Liste der Dictionaries aus run_clustering_comparison()
                            (jedes enthält 'algorithm', 'k', 'silhouette', 'labels', ...)
    
    Returns:
        dict: das beste (gefilterte) Ergebnis-Dictionary
    """
    if not comparison_results:
        raise ValueError("Keine Clustering-Ergebnisse vorhanden")
    
    # Gesamtzahl der Mitarbeiter (aus erstem Ergebnis)
    n_total = len(comparison_results[0]['labels'])
    
    # Mindestgröße pro Cluster: mind. 2% oder mind. 3 Samples
    min_cluster_size = max(3, int(n_total * 0.02))
    print(f"  Mindest-Clustergröße für gültige Lösung: {min_cluster_size} Mitarbeiter")
    
    valid = []
    
    for r in comparison_results:
        labels = r['labels']
        # Nur nicht-Rauschen-Cluster zählen (für HDBSCAN)
        cluster_counts = pd.Series(labels[labels != -1]).value_counts()
        
        if len(cluster_counts) == 0:
            continue
            
        min_size = cluster_counts.min()
        
        if min_size >= min_cluster_size:
            valid.append(r)
        else:
            print(f"    Überspringe {r['algorithm']} k={r['k'] or 'auto'}: "
                  f"kleinstes Cluster hat nur {min_size} (min={min_cluster_size})")
    
    # Fallback: wenn nichts übrig bleibt, nimm trotzdem das Beste
    if not valid:
        print("  Warnung: Keine Konfiguration erfüllt Mindestgröße → nehme beste ohne Filter")
        valid = comparison_results
    
    # Beste nach Silhouette-Score
    best = max(valid, key=lambda r: r['silhouette'])
    
    print(f"\n   Beste gültige Konfiguration: "
          f"{best['algorithm']} "
          f"(k={best['k'] or 'auto'}) "
          f"Silhouette = {best['silhouette']:.4f}")
    
    if 'noise_count' in best:
        print(f"   (Rauschen: {best['noise_count']} Punkte)")
    
    return best

## SCHRITT 6: Cluster-Charakterisierung & Benennung

In [16]:
import pandas as pd
import numpy as np

def characterize_clusters(feature_df: pd.DataFrame, labels: np.ndarray, feature_cols: list) -> tuple:
    """
    Berechne Cluster-Profile (Mittelwerte) und vergib sprechende Namen
    basierend auf den dominanten Charakteristika jedes Clusters.
    
    Args:
        feature_df:   Original DataFrame mit Features (eine Zeile pro Mitarbeiter)
        labels:       Array mit Cluster-Zuweisungen (z. B. aus best_config['labels'])
        feature_cols: Liste der verwendeten Feature-Spalten
    
    Returns:
        tuple: (cluster_names: dict {cluster_id: name}, profile: pd.DataFrame mit Mittelwerten)
    """
    print("Characterizing clusters...")
    
    # Arbeitskopie nur mit relevanten Spalten + Cluster-Label
    temp_df = feature_df[feature_cols].copy()
    temp_df['_cluster'] = labels
    
    # Rauschen (-1) bei HDBSCAN ignorieren
    valid = temp_df[temp_df['_cluster'] != -1].copy()
    
    if valid.empty:
        print("Warnung: Keine gültigen Cluster-Zuweisungen (nur Rauschen?)")
        return {}, pd.DataFrame()
    
    # Mittelwerte pro Cluster
    profile = valid.groupby('_cluster')[feature_cols].mean()
    
    # Normalisierte Werte (0–1) → für relative Vergleiche
    profile_norm = (profile - profile.min()) / (profile.max() - profile.min() + 1e-9)
    
    cluster_names = {}
    
    for cluster_id, row in profile_norm.iterrows():
        raw = profile.loc[cluster_id]
        
        # Wichtige Kennzahlen (normalisiert)
        ticket_vol   = row.get('ticket_count',       0)
        time_eff     = 1 - row.get('median_time_hours', 0)   # hoch = schneller
        quality      = row.get('success_rate',       0)
        reopen       = row.get('reopen_rate',        0)
        first_touch  = row.get('first_touch_rate',   0)
        hd_service   = row.get('pct_hd_service',     0)
        high_prio    = row.get('pct_high',           0)
        steps        = row.get('avg_steps',          0)
        comments     = row.get('avg_comments',       0)
        
        # ─── Regelbasierte, sprechende Namen ───
        if ticket_vol >= 0.7 and time_eff >= 0.6:
            name = "Volumen-Performer"
        elif high_prio >= 0.6 and steps >= 0.6:
            name = "Eskalations-Spezialist"
        elif hd_service >= 0.6:
            name = "HD-Service-Spezialist"
        elif reopen >= 0.6 and quality < 0.4:
            name = "Problemlöser (hohe Reopens)"
        elif quality >= 0.7 and ticket_vol < 0.4:
            name = "Qualitäts-Fokus"
        elif first_touch >= 0.7 and time_eff >= 0.5:
            name = "Erstlöser"
        elif comments >= 0.7:
            name = "Kommunikations-Intensiv"
        elif ticket_vol < 0.3:
            name = "Spezialisten"
        else:
            name = "Allrounder"
        
        cluster_names[cluster_id] = name
    
    # ─── Duplikate vermeiden (z. B. zwei "Allrounder") ───
    seen = {}
    for cid, name in cluster_names.items():
        if name in seen:
            seen[name] += 1
            cluster_names[cid] = f"{name} {seen[name]}"
        else:
            seen[name] = 1
    
    # ─── Ausgabe ───
    print("\n  Cluster-Namen & Schlüsselkennzahlen:")
    for cid, name in cluster_names.items():
        mask = (labels == cid)
        count = mask.sum()
        raw = profile.loc[cid]
        
        print(f"    Cluster {cid:2d} → '{name}' ({count} Mitarbeiter)")
        print(f"      tickets={raw.get('ticket_count',0):4.0f}, "
              f"median_time_h={raw.get('median_time_hours',0):5.1f}, "
              f"success={raw.get('success_rate',0):5.1%}, "
              f"reopen={raw.get('reopen_rate',0):5.1%}, "
              f"first_touch={raw.get('first_touch_rate',0):5.1%}")
    
    return cluster_names, profile
    

## STEP 7: Merge Everything & Save

In [17]:
import pandas as pd
import joblib
from pathlib import Path

def run():
    print("=" * 65)
    print("EMPLOYEE CLUSTERING — LEAKAGE-FREE UNSUPERVISED ANALYSIS")
    print("=" * 65)

    # ─── 1. Daten laden ───
    print(f"\n[1/7] Lade Daten: {DATA_RAW}")
    df = pd.read_csv(DATA_RAW, low_memory=False)
    print(f"      Rohdaten: {df.shape[0]:,} rows, {df.shape[1]} columns")

    # ─── 2. Feature Engineering ───
    print("\n[2/7] Feature Engineering...")
    feature_df = build_features(df)

    feature_cols = [
        'ticket_count', 'median_time_hours', 'std_time_hours',
        'avg_steps', 'avg_comments', 'reopen_rate', 'success_rate',
        'first_touch_rate', 'pct_high', 'pct_low', 'pct_hd_service',
    ]

    # ─── 3. Preprocessing ───
    print("\n[3/7] Preprocessing (RobustScaler, NaN-Fill, Zero-Variance-Drop)...")
    X_scaled, scaler, active_cols = preprocess(feature_df, feature_cols)

    # ─── 4. Dimensionsreduktion ───
    print("\n[4/7] Dimensionsreduktion...")
    dim = reduce_dimensions(X_scaled)

    # ─── 5. Clustering-Vergleich ───
    print("\n[5/7] Clustering-Vergleich...")
    df_comparison, all_results = run_clustering_comparison(X_scaled)

    # ─── 6. Beste Konfiguration auswählen ───
    print("\n[6/7] Auswahl beste Konfiguration...")
    best = select_best(all_results)
    best_labels = best['labels']
    best_algo = best['algorithm']
    best_k = best['k']
    best_silhouette = best['silhouette']
    best_model = best['model']

    # ─── 7. Cluster charakterisieren & Ergebnisse speichern ───
    print("\n[7/7] Cluster-Charakterisierung & Speichern...")
    cluster_names_map, profile_df = characterize_clusters(
        feature_df, best_labels, active_cols
    )

    # Koordinaten auswählen (UMAP bevorzugt, sonst PCA)
    coords = dim['umap'] if dim.get('use_umap', False) else dim['pca']
    coord_prefix = 'umap' if dim.get('use_umap', False) else 'pca'

    # Finales DataFrame bauen
    cluster_df = feature_df.copy()
    cluster_df['cluster'] = best_labels
    cluster_df['cluster_name'] = cluster_df['cluster'].map(
        lambda x: cluster_names_map.get(x, 'Rauschen')
        if x != -1 else 'Rauschen (Outlier)'
    )
    cluster_df[f'{coord_prefix}_1'] = coords[:, 0]
    cluster_df[f'{coord_prefix}_2'] = coords[:, 1]
    cluster_df.rename(columns={'issue_assignee': 'employee'}, inplace=True)

    # Einheitliche Spaltennamen für Streamlit / Visualisierung
    if coord_prefix == 'pca':
        cluster_df['umap_1'] = cluster_df['pca_1']
        cluster_df['umap_2'] = cluster_df['pca_2']

    # ─── Speichern ───
    cluster_csv = DATA_PROCESSED / "employee_clusters.csv"
    cluster_df.to_csv(cluster_csv, index=False)
    print(f"   Cluster-CSV: {cluster_csv} ({len(cluster_df)} Mitarbeiter)")

    profile_valid = profile_df.reset_index()
    profile_valid['cluster_name'] = profile_valid['_cluster'].map(cluster_names_map)
    profile_valid.drop(columns=['_cluster'], inplace=True)

    profile_csv = DATA_PROCESSED / "cluster_profiles.csv"
    profile_valid.to_csv(profile_csv, index=False)
    print(f"   Profile-CSV: {profile_csv}")

    comparison_csv = DATA_PROCESSED / "clustering_comparison.csv"
    df_comparison.to_csv(comparison_csv, index=False)
    print(f"   Vergleichs-CSV: {comparison_csv}")

    model_path = MODELS_DIR / "employee_clustering.joblib"
    joblib.dump({
        'scaler': scaler,
        'model': best_model,
        'feature_cols': active_cols,
        'cluster_names': cluster_names_map,
        'algorithm': best_algo,
        'n_clusters': best_k,
        'silhouette': best_silhouette,
        'davies_bouldin': best['davies_bouldin'],
        'calinski_harabasz': best['calinski_harabasz'],
        'coord_prefix': coord_prefix,
        'dim_reduction': dim,
        'comparison_df': df_comparison,
    }, model_path)
    print(f"   Modell: {model_path}")

    # ─── Finale Zusammenfassung ───
    print("\n" + "=" * 65)
    print("ERGEBNIS-ZUSAMMENFASSUNG")
    print("=" * 65)
    print(f"  Mitarbeiter analysiert : {len(cluster_df)}")
    print(f"  Bester Algorithmus     : {best_algo}")
    print(f"  Beste Cluster-Anzahl   : {best_k}")
    print(f"  Silhouette Score       : {best_silhouette:.4f}")
    print(f"  Davies-Bouldin Score   : {best['davies_bouldin']:.4f}")
    print(f"  Calinski-Harabasz      : {best['calinski_harabasz']:.2f}")
    print(f"  Dimensionsreduktion    : {'UMAP' if dim.get('use_umap') else 'PCA'}")

    print("\n  Cluster-Verteilung:")
    vc = cluster_df['cluster_name'].value_counts()
    for name, count in vc.items():
        pct = count / len(cluster_df) * 100
        print(f"    {name:<30} {count:>4} Mitarbeiter ({pct:.1f}%)")

    print("=" * 65)

    return cluster_df, df_comparison

##  Execution

In [18]:
run()

EMPLOYEE CLUSTERING — LEAKAGE-FREE UNSUPERVISED ANALYSIS

[1/7] Lade Daten: /home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw/issues_snapshot.csv


FileNotFoundError: [Errno 2] No such file or directory: '/home/openclaw/.openclaw/workspace/projects/Employee performance 5/notebooks/data/raw/issues_snapshot.csv'